# Block 09: The Transformer block: FFN, residuals, LayerNorm, dropout

**Paper anchor:** §3.1, §3.3 and Eq. 2, §5.4 (residual dropout)
**Builds on:** blocks 07 and 08; prerequisite P7
**Time:** about 5 hours, in short sessions
**You will write:** `layer_norm`, `ffn` (NumPy); `FeedForward`, `TransformerBlock` (PyTorch)
**Done when:** all three checkpoints pass

> **How this notebook works.** Claude teaches each concept in chat, one step at a time (type `next`).
> You write every line of solution code, in the cells marked **YOUR CODE**. Checkpoint cells test behavior only
> and never contain a solution (please don't open `checks/` until you pass). Stuck or wrong? Paste your code and
> the output into chat: we find the misconception together. Log every mistake in the table at the bottom.

In [ ]:
# provided: repo paths (works from notebooks/ locally and from the repo root on Colab)
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
np.set_printoptions(precision=4, suppress=True)
from checks import b09

## Big picture
Each encoder layer is two sub-layers, each wrapped as $\mathrm{LayerNorm}(x + \mathrm{Sublayer}(x))$ (§3.1):
multi-head self-attention, then a position-wise feed-forward network
$\mathrm{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$ (Eq. 2, $d_{ff} = 2048$). Dropout hits each sub-layer output before
the residual add (§5.4).

**Paper vs. later practice.** The paper is post-norm. GPT-2 and most later LLMs use pre-norm,
$x + \mathrm{Sublayer}(\mathrm{LayerNorm}(x))$ (outside knowledge). This repo supports both through `norm_first`,
so block 13 can compare them.

## Before you code
Answer in the cell below, in your own words. Guessing is fine; we check in chat.

1. Why must every sub-layer output exactly d_model dimensions?
2. 'Position-wise' FFN: write it as a single matrix product over a (B·T, d_model) array.
3. What problem does a residual connection solve? (Outside knowledge: He et al. [11].)

*My answers:*

1. 
2. 
3. 

## Part A: NumPy

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

def layer_norm(x, gamma, beta, eps=1e-5):
    """Layer normalization over the last axis (Ba et al. [1]).

    x: (..., d); gamma, beta: (d,)
    returns gamma * (x - mean) / sqrt(var + eps) + beta, with mean and population variance
    taken over the last axis of each position. eps = 1e-5 is an assumption (the paper gives none).
    """
    raise NotImplementedError("Block 09: your turn")

def ffn(x, W1, b1, W2, b2):
    """Position-wise feed-forward network, Eq. 2: max(0, x W1 + b1) W2 + b2.

    x: (..., d_model); W1: (d_model, d_ff); b1: (d_ff,); W2: (d_ff, d_model); b2: (d_model,)
    returns (..., d_model)
    """
    raise NotImplementedError("Block 09: your turn")

In [ ]:
b09.check_layer_norm(layer_norm)
b09.check_ffn(ffn)

## Part B: PyTorch

In [ ]:
import torch, torch.nn as nn
from minitransformer.attention import MultiHeadAttention  # graduated in block 07

In [ ]:
# YOUR CODE: replace each `raise NotImplementedError` with your implementation

class FeedForward(nn.Module):
    """Eq. 2 position-wise FFN: lin2(dropout(relu(lin1(x)))).

    __init__(d_model, d_ff, dropout=0.0)
        Attributes: lin1 = nn.Linear(d_model, d_ff), lin2 = nn.Linear(d_ff, d_model) (with biases, as in Eq. 2).
        Dropout inside the FFN is an assumption (tensor2tensor style); residual dropout lives in the block.
    forward(x): (B, T, d_model) -> (B, T, d_model)
    """

    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        raise NotImplementedError("Block 09: your turn")

    def forward(self, x):
        raise NotImplementedError("Block 09: your turn")

class TransformerBlock(nn.Module):
    """One encoder-style layer: self-attention sub-layer, then FFN sub-layer (§3.1).

    __init__(d_model, n_heads, d_ff, dropout=0.1, norm_first=False)
        Attributes: attn (MultiHeadAttention), ffn (FeedForward), ln1, ln2 (nn.LayerNorm(d_model)).
        norm_first=False is the paper's post-norm:  x = LayerNorm(x + Dropout(Sublayer(x)))
        norm_first=True is pre-norm (outside the paper, used by GPT-2 and later):
                                                    x = x + Dropout(Sublayer(LayerNorm(x)))
        Residual dropout on each sub-layer output before the add (§5.4).
    forward(x, mask=None): (B, T, d_model) -> (B, T, d_model); mask is passed to self-attention.
    """

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, norm_first=False):
        super().__init__()
        raise NotImplementedError("Block 09: your turn")

    def forward(self, x, mask=None):
        raise NotImplementedError("Block 09: your turn")

In [ ]:
b09.check_torch_block(TransformerBlock)

## Explain it back
Without notes. Then paste your answers into chat.

1. Draw one encoder layer from Fig. 1 and label every shape.
2. Post-norm vs pre-norm: write both formulas and say which one the paper uses.
3. Where exactly is dropout applied in §5.4, and what does the paper leave unsaid?

*My explanation:*

1. 
2. 
3. 

## Mistakes log
Every wrong attempt is data. Copy finished rows into `journal/mistakes.md`.

| # | What I wrote | Symptom | The misconception | The fix |
|---|---|---|---|---|
| 1 | | | | |

## Graduate
NumPy functions into `numpy_ops.py`; classes into `src/minitransformer/layers.py`; `pytest -q`; commit `block 09: transformer block`. Mark P7 done.